# Binance Testnet API test

In [4]:
import ccxt
import time

# --- TES CLÉS TESTNET BINANCE FUTURES ---
API_KEY = 'dnB7NFqyYYIcFHwi2ca8PsQ8wHeRrQ7IoFXmIRmZVdiwJAQfD4VOs193gIZqAjL8'
API_SECRET = 'mVXxrX5K0BBqLaDQ3sIRNHbHi1x5MuSSoCv9yevZumHziQYesrMA4LB6Tlcy521R'

def run_plumbing_test():
    print("="*60)
    print("🔧 PLUMBING TEST : BINANCE DEMO TRADING (FUTURES)")
    print("="*60)

    # 1. Connexion au nouveau Demo Trading via CCXT
    exchange = ccxt.binance({
        'apiKey': API_KEY,
        'secret': API_SECRET,
        'enableRateLimit': True,
        'options': {
            'defaultType': 'future', 
            'adjustForTimeDifference': True  # 🚨 AJOUTE CETTE LIGNE
        }
    })
    
    # Activation du nouveau mode
    exchange.enable_demo_trading(True)

    try:
        # 3. Test de lecture des prix via le Carnet d'Ordres (Order Book)
        print("\n2️⃣ Vérification des prix (Carnet d'ordres AAVE et ETH)...")
        
        # On télécharge les 5 meilleures offres d'achat et de vente
        aave_ob = exchange.fetch_order_book('AAVE/USDT', limit=5)
        eth_ob = exchange.fetch_order_book('ETH/USDT', limit=5)
        
        # Extraction du meilleur acheteur (Bid) et vendeur (Ask)
        aave_bid = aave_ob['bids'][0][0] if aave_ob['bids'] else 0
        aave_ask = aave_ob['asks'][0][0] if aave_ob['asks'] else 0
        
        eth_bid = eth_ob['bids'][0][0] if eth_ob['bids'] else 0
        eth_ask = eth_ob['asks'][0][0] if eth_ob['asks'] else 0
        
        print(f"   AAVE : Bid {aave_bid} | Ask {aave_ask} | Spread : {aave_ask - aave_bid:.3f} USDT")
        print(f"   ETH  : Bid {eth_bid} | Ask {eth_ask} | Spread : {eth_ask - eth_bid:.3f} USDT")

        # --- DANGER ZONE : PASSAGE D'ORDRES (MOCK) ---
        print("\n3️⃣ Test de passage d'ordre (Mode Simulation - Non Exécuté)")
        # On calcule la taille avec le vrai prix d'exécution (Ask pour acheter, Bid pour vendre)
        aave_size = round(50 / aave_ask, 1) if aave_ask else 0
        eth_size = round(50 / eth_bid, 3) if eth_bid else 0
        
        print(f"   Taille calculée pour 50$ -> AAVE : {aave_size} | ETH : {eth_size}")
        print("✅ Logique de calcul des tailles et lecture du carnet d'ordres validée.")

        # 4. Vérification des limites d'API (Rate Limits)
        print("\n4️⃣ Ping de l'API pour vérifier le blocage (Rate Limits)...")
        for i in range(5):
            exchange.fetch_ticker('BTC/USDT')
            print(f"   Ping {i+1}/5 OK")
            time.sleep(0.1) # Rafale rapide pour tester
        print("✅ Aucun blocage de l'API détecté.")

    except ccxt.AuthenticationError:
        print("\n❌ ERREUR : Clés API invalides. Vérifie que ce sont bien celles du DEMO TRADING.")
    except Exception as e:
        print(f"\n❌ ERREUR INCONNUE : {e}")

    print("\n" + "="*60)
    print("🏁 TEST TERMINÉ")
    print("="*60)

if __name__ == "__main__":
    run_plumbing_test()

🔧 PLUMBING TEST : BINANCE DEMO TRADING (FUTURES)

2️⃣ Vérification des prix (Carnet d'ordres AAVE et ETH)...
   AAVE : Bid 74.8 | Ask 75.08 | Spread : 0.280 USDT
   ETH  : Bid 1652.74 | Ask 1653.56 | Spread : 0.820 USDT

3️⃣ Test de passage d'ordre (Mode Simulation - Non Exécuté)
   Taille calculée pour 50$ -> AAVE : 0.7 | ETH : 0.03
✅ Logique de calcul des tailles et lecture du carnet d'ordres validée.

4️⃣ Ping de l'API pour vérifier le blocage (Rate Limits)...
   Ping 1/5 OK
   Ping 2/5 OK
   Ping 3/5 OK
   Ping 4/5 OK
   Ping 5/5 OK
✅ Aucun blocage de l'API détecté.

🏁 TEST TERMINÉ


In [5]:
import ccxt
import time

# --- TES CLÉS DEMO TRADING BINANCE ---
API_KEY = 'dnB7NFqyYYIcFHwi2ca8PsQ8wHeRrQ7IoFXmIRmZVdiwJAQfD4VOs193gIZqAjL8'
API_SECRET = 'mVXxrX5K0BBqLaDQ3sIRNHbHi1x5MuSSoCv9yevZumHziQYesrMA4LB6Tlcy521R'

def run_live_fire_test():
    print("="*70)
    print("🔥 EXÉCUTION RÉELLE : BAPTÊME DU FEU SUR LE DEMO TRADING")
    print("="*70)

    exchange = ccxt.binance({
        'apiKey': API_KEY,
        'secret': API_SECRET,
        'enableRateLimit': True,
        'options': {
            'defaultType': 'future', 
            'adjustForTimeDifference': True
        }
    })
    exchange.enable_demo_trading(True)
    exchange.load_markets()

    sym1, sym2 = 'AAVE/USDT', 'ETH/USDT'

    try:
        # --- 1. CONFIGURATION DU LEVIER ---
        print("\n1️⃣ Configuration du levier (1x pour la sécurité)...")
        exchange.set_leverage(1, sym1)
        exchange.set_leverage(1, sym2)
        print("✅ Leviers configurés à 1x.")

        # --- 2. CALCUL DES TAILLES ---
        print("\n2️⃣ Lecture du carnet d'ordres...")
        aave_ob = exchange.fetch_order_book(sym1, limit=5)
        eth_ob = exchange.fetch_order_book(sym2, limit=5)
        
        aave_ask = aave_ob['asks'][0][0] # Prix pour ACHETER
        eth_bid = eth_ob['bids'][0][0]   # Prix pour VENDRE
        
        # On va trader ~50$ sur chaque patte
        aave_size = round(50 / aave_ask, 1)
        eth_size = round(50 / eth_bid, 3)
        print(f"   Préparation des lots : AAVE={aave_size} | ETH={eth_size}")

        # --- 3. L'ENTRÉE (ENTRY) ---
        print("\n3️⃣ ENTRÉE EN POSITION (Ouverture du Spread)...")
        print(f"   -> Envoi de l'ordre : LONG {sym1}")
        order_aave_entry = exchange.create_market_buy_order(sym1, aave_size)
        
        print(f"   -> Envoi de l'ordre : SHORT {sym2}")
        order_eth_entry = exchange.create_market_sell_order(sym2, eth_size)

        print("✅ Positions ouvertes !")
        
        # --- 4. ANALYSE DU REÇU (FEES & PRIX D'EXÉCUTION) ---
        time.sleep(2) # On laisse 2 secondes à Binance pour confirmer
        
        # On va chercher le "vrai" prix auquel l'ordre est passé
        aave_fill_price = exchange.fetch_order(order_aave_entry['id'], sym1)['average']
        eth_fill_price = exchange.fetch_order(order_eth_entry['id'], sym2)['average']
        
        print("\n📊 REÇU D'EXÉCUTION (Entry) :")
        print(f"   AAVE : Prix demandé {aave_ask} -> Prix obtenu {aave_fill_price}")
        print(f"   ETH  : Prix demandé {eth_bid} -> Prix obtenu {eth_fill_price}")

        # --- 5. ATTENTE ET SORTIE (EXIT) ---
        print("\n⏳ Maintien de la position pendant 10 secondes pour observer le P&L sur le site Binance...")
        time.sleep(10)

        print("\n4️⃣ SORTIE DE POSITION (Fermeture du Spread)...")
        print(f"   -> Envoi de l'ordre de fermeture : SELL {sym1}")
        exchange.create_market_sell_order(sym1, aave_size)
        
        print(f"   -> Envoi de l'ordre de fermeture : BUY {sym2}")
        exchange.create_market_buy_order(sym2, eth_size)

        print("✅ Positions fermées avec succès. Portefeuille nettoyé.")

    except ccxt.InsufficientFunds as e:
        print(f"\n❌ ERREUR DE FONDS : Tu n'as pas assez d'USDT sur le Testnet. ({e})")
    except ccxt.InvalidOrder as e:
        print(f"\n❌ ERREUR D'ORDRE : Taille trop petite ou problème de paramètres. ({e})")
    except Exception as e:
        print(f"\n❌ ERREUR INCONNUE : {e}")

    print("\n" + "="*70)
    print("🏁 EXERCICE À BALLES RÉELLES TERMINÉ")
    print("="*70)

if __name__ == "__main__":
    run_live_fire_test()

🔥 EXÉCUTION RÉELLE : BAPTÊME DU FEU SUR LE DEMO TRADING

1️⃣ Configuration du levier (1x pour la sécurité)...
✅ Leviers configurés à 1x.

2️⃣ Lecture du carnet d'ordres...
   Préparation des lots : AAVE=0.7 | ETH=0.03

3️⃣ ENTRÉE EN POSITION (Ouverture du Spread)...
   -> Envoi de l'ordre : LONG AAVE/USDT
   -> Envoi de l'ordre : SHORT ETH/USDT
✅ Positions ouvertes !

📊 REÇU D'EXÉCUTION (Entry) :
   AAVE : Prix demandé 74.45 -> Prix obtenu 74.45
   ETH  : Prix demandé 1648.76 -> Prix obtenu 1648.76

⏳ Maintien de la position pendant 10 secondes pour observer le P&L sur le site Binance...

4️⃣ SORTIE DE POSITION (Fermeture du Spread)...
   -> Envoi de l'ordre de fermeture : SELL AAVE/USDT
   -> Envoi de l'ordre de fermeture : BUY ETH/USDT
✅ Positions fermées avec succès. Portefeuille nettoyé.

🏁 EXERCICE À BALLES RÉELLES TERMINÉ


Notes

1. Le "Funding Rate" (Le coût du temps)
Ce que c'est : Sur Binance Futures, pour maintenir le prix des contrats aligné avec le prix spot, les traders se paient entre eux toutes les 8 heures. Si tout le monde est Long sur AAVE, les Longs paient une taxe aux Shorts.

L'impact sur notre bot : Puisque notre Demi-Vie (Half-Life) est de 27 heures, notre bot va traverser environ 3 ou 4 paiements de Funding. Si on est Short sur la crypto que tout le monde veut Shorter, on va payer cette taxe.

Ce qu'il faut dire aux recruteurs : "La rentabilité brute du spread couvre largement les frais d'exécution (0.20%), mais la performance nette finale dépendra de l'accumulation des Funding Rates pendant la période de détention de 24-48h."

2. L'Illusion du Slippage (Retail vs Institutionnel)
Ce que ton Live Test a prouvé : Tu as tradé 50$ par patte, et le prix demandé a été exactement le prix obtenu. Le Slippage a été de 0.00%. La liquidité de AAVE et ETH a absorbé ton ordre instantanément.

La limite : "Cette stratégie est ultra-rentable avec 1 000$ ou 10 000$. Mais si j'entre avec 5 millions de dollars au marché, mon propre ordre va faire bouger le prix d'AAVE, et je vais détruire le spread avant même d'avoir pu l'acheter."

3. La Survie aux "Fat Tails" (Le Kurtosis)
Ce que les statistiques ont prouvé : Notre test a montré un Kurtosis élevé avec un Z-Score max à 5.76 lors d'un événement inattendu.

La conclusion architecturale : "Une stratégie naïve 'All-In à Z=2' finirait par être liquidée par un cygne noir. J'ai donc conçu la stratégie pour utiliser un 'DCA Spatial' (entrée en grille fractionnée) ou un filtre Machine Learning pour ne pas acheter un spread qui s'élargit pour une raison fondamentale légitime."

4. Le Ratio Timeframe / Rentabilité
Ce que le Radar a prouvé : Sur du 1 minute, les écarts sont trop petits (0.05%) et sont mangés par les frais Binance. Sur du Daily, les fondamentaux des projets changent et la cointégration se brise.

La conclusion : "Le Timeframe 15m / 1H est le 'Sweet Spot' quantitatif. Il offre une amplitude de spread (~6%) suffisamment large pour rendre les frais de transaction de 0.20% négligeables."

5. Le Hedge Parfait (La Neutralité au Marché)
Ce que ta capture d'écran a prouvé : Avoir 50$ en Long et 50$ en Short crée un Delta Neutre (Beta de 0).

L'argument massue : "Contrairement à ma Phase 1 et Phase 2 qui tentaient de deviner la direction du Bitcoin, cette Phase 3 est immunisée contre le risque de marché (Market Risk). Si le marché crypto perd 30% en une nuit, la perte sur ma patte Long est mathématiquement annulée par le gain sur ma patte Short."